# Vector stores and semantic search



In [15]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import torch

## Part I: Basic vector store implementation

In [16]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        # Listas para almacenar los documentos y sus vectores numéricos
        self.documents: list[Document] = []
        self.embeddings: list[torch.Tensor] = []

    def add_documents(self, documents: list[Document]):
        # Guardar los objetos Document
        self.documents.extend(documents)

        print(f"Se han agregado {len(self.documents)} documentos.")

        texts : list[str] = []
        
        for doc in self.documents:
            texts.append(doc.text)

        new_embeddings = self.embedding_model.encode(texts)

        self.embeddings.extend(new_embeddings)
        
        print(f"Total de embeddings generados: {len(self.embeddings)}")

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if not self.documents:
            return []
        
        # Convertir la consulta en un embedding
        query_embedding = self.embedding_model.encode([query])

        # Calcular la similitud coseno entre la query y todos los documentos
        cosine_scores = util.cos_sim(query_embedding, self.embeddings)[0]

        results = []
        # score.item() extrae el valor numérico (float) del tensor de PyTorch
        for index, score in enumerate(cosine_scores):
            results.append(SearchResult(score=score.item(), document=self.documents[index]))

        # Ordenar los resultados de mayor a menor score
        results.sort(key=lambda x: x.score, reverse=True)
        
        # Retornar los top_k resultados
        return results[:top_k]
        

## Part II: Filtering by metadata

In [17]:
class FilteredVectorStore(VectorStore):
    def __init__(self, embedding_model: SentenceTransformer):
        super().__init__(embedding_model=embedding_model)

    def add_documents(self, documents: list[Document]):
        super().add_documents(documents=documents)

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        if not self.documents:
            return []

        # Convertir la consulta en un embedding
        query_embedding = self.embedding_model.encode([query])

        # Calcular la similitud coseno entre la query y todos los documentos
        cosine_scores = util.cos_sim(query_embedding, self.embeddings)[0]

        results = []
        # score.item() extrae el valor numérico (float) del tensor de PyTorch
        for index, score in enumerate(cosine_scores):

            doc = self.documents[index]

            # Lógica de filtrado por metadatos
            if metadata_filter is not None:
                match = True
                for key, value in metadata_filter.items():
                    # Si la clave no está en el documento o el valor no es igual, se descarta
                    if doc.metadata.get(key) != value:
                        match = False
                        break
                
                # Si el documento no cumple con el filtro, saltamos a la siguiente iteración
                if not match:
                    continue
                
            results.append(SearchResult(score=score.item(), document=doc))

        # Ordenar los resultados de mayor a menor score
        results.sort(key=lambda x: x.score, reverse=True)
        
        # Retornar los top_k resultados
        return results[:top_k]
        

## Bases de Datos Vectoriales de Animales

En esta sección se implementa un **VectorStore** desde cero para realizar búsquedas semánticas utilizando lenguaje natural. Para ello, se emplea el **Animal Fun Facts Dataset**.

**Proceso implementado:**
1. **Carga y estructuración de datos:** Se extraen los datos del archivo CSV de [animal-fun-facts-dataset](https://github.com/ekohrt/animal-fun-facts-dataset) y se iteran para crear instancias de la clase `Document`. La columna de texto principal se utiliza como el contenido del documento, mientras que `animal_name`, `source`, `media_link` y `wikipedia_link` se almacenan como un diccionario de metadatos. En total, se cargaron con éxito 7,734 documentos.
2. **Generación de Embeddings:** Utilizando el modelo `all-MiniLM-L6-v2` de `SentenceTransformer`, se calculan los vectores densos (embeddings) para cada uno de los textos.
3. **Búsqueda Semántica:** Se realizan 5 consultas de ejemplo (por ejemplo, *"Animals with unusual sleeping habits or patterns"* o *"Facts about marine life and deep sea creatures"*). El sistema calcula la similitud del coseno entre el vector de la consulta y los de los documentos, devolviendo los resultados más relevantes junto con su *score*, texto y metadatos asociados.

In [18]:
file_path = '../data/NLP/animal-fun-facts-dataset.csv'
df : pd.DataFrame = None

try:
    # Cargar los datos al DataFrame
    df = pd.read_csv(file_path)

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An unexpected error occurred while reading the CSV: {e}")

# La lista vacia de animales
documentos : list[Document] = []

# Llenar esta lista
for index, row in df.iterrows():
    text_content = str(row.get('text'))
    metadata = {
        'animal_name' : str(row.get('animal_name')),
        'source' : str(row.get('source')),
        'media_link' : str(row.get('media_link')),
        'wikipedia_link' : str(row.get('wikipedia_link'))
    }

    documentos.append(Document(text=text_content, metadata=metadata))

print(f"Successfully loaded {len(documentos)} documents.")

Successfully loaded 7734 documents.


In [19]:
# Cargar el modelo de Sentence Transformers
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Crear la instancia de VectorStore
vector_store = VectorStore(embedding_model=embedding_model)

# Agregar los documentos a la vector store
vector_store.add_documents(documents=documentos)

# Escribir 5 consultas de animales
consultas = [
    "Animals with unusual sleeping habits or patterns",
    "Fascinating traits of birds and flying creatures",
    "Which animals are known for their extreme speed or agility?",
    "Facts about marine life and deep sea creatures",
    "Information about the diet and hunting mechanisms of predators"
]

# Ejecutamos las búsquedas y mostramos los resultados
for i, query in enumerate(consultas, 1):
    print(f"\n--- Consulta {i}: '{query}' ---")
    
    # Buscamos los 5 documentos más relevantes
    resultados = vector_store.search(query)
    
    for rank, result in enumerate(resultados, 1):
        print(f"  Resultado {rank}:")
        # Mostramos Score, Texto y Metadatos
        print(f"    * Score: {result.score:.4f}")
        print(f"    * Texto: {result.document.text}")
        print(f"    * Metadatos: {result.document.metadata}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Se han agregado 7734 documentos.
Total de embeddings generados: 7734

--- Consulta 1: 'Animals with unusual sleeping habits or patterns' ---
  Resultado 1:
    * Score: 0.6527
    * Texto: These animals are diurnal, sleeping in treetop leaves and branches during the night. They spend most of day in search of food, grooming, and resting.
    * Metadatos: {'animal_name': 'coatimundi', 'source': 'https://seaworld.org/animals/facts/mammals/coatimundi/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Coati'}
  Resultado 2:
    * Score: 0.6165
    * Texto: They sleep during the day and are awake at night
    * Metadatos: {'animal_name': 'syrian hamster', 'source': 'https://www.animalfactsencyclopedia.com/Syrian-hamster.html', 'media_link': 'nan', 'wikipedia_link': '/wiki/Golden_hamster'}
  Resultado 3:
    * Score: 0.5722
    * Texto: They are nocturnal to help them escape high temperatures..
They spend all day in burrows they make in the sand, only coming out at night when it is cooler.
    

## Bases de Datos Vectoriales Filtradas de Criterios de Elegibilidad para Ensayos Clínicos

En esta sección se extiende la funcionalidad anterior para implementar un **FilteredVectorStore**, el cual permite refinar las búsquedas aplicando condiciones exactas sobre los metadatos de los documentos. Para este ejercicio se utiliza el conjunto de datos [Clinical Trial Eligibility Criteria de Kaggle](https://www.kaggle.com/datasets/harrachimustapha/clinical-trial-eligibility-criteria-dataset).

**Proceso implementado:**
1. **Carga de Datos Clínicos:** Se procesa el archivo `trials_clean.csv`. La columna `combined_text_for_retrieval` se asigna como el texto base de búsqueda, mientras que el resto de las columnas (condiciones, estado, intervenciones, tipo de estudio, etc.) se almacenan como metadatos. Se vectorizaron exitosamente 60,337 documentos.
2. **Búsqueda Semántica con Filtros:** El método de búsqueda fue sobrescrito para aceptar el parámetro `metadata_filter`. Esto asegura que los resultados devueltos no solo tengan un alto puntaje de similitud semántica con consultas como *"treatment studies for breast cancer in women"*, sino que cumplan estrictamente con las reglas dictadas en el diccionario de metadatos (por ejemplo, filtrar por fase de prueba, tipo de enfermedad o género) descartando cualquier documento que no coincida.

In [20]:
file_path_clinical_trial = '../data/NLP/trials_clean.csv'
df_clinical_trial : pd.DataFrame = None

try:
    df_clinical_trial = pd.read_csv(file_path_clinical_trial)

except FileNotFoundError:
    print(f"Error: The file '{file_path_clinical_trial}' was not found. Please check the path.")
except Exception as e:
    print(f"An unexpected error occurred while reading the CSV: {e}")

# La lista vacia de ensayos clínicos
documentos_clinical_trial : list[Document] = []

# Llenar esta lista 
for index, row in df_clinical_trial.iterrows():
    texto_completo = str(row.get('combined_text_for_retrieval'))
    metadatos = {}
    for columna, valor in row.items():
        if columna not in ["combined_text_for_retrieval"]:
            metadatos[columna] = str(valor)

    documentos_clinical_trial.append(Document(text=texto_completo, metadata=metadatos))

print(f"Successfully loaded {len(documentos_clinical_trial)} documents.")

Successfully loaded 60337 documents.


In [21]:
# Crear la instancia de FilteredVectorStore
filtered_vector_store_clinical_trial  = FilteredVectorStore(embedding_model=embedding_model)

# Agregar los documentos a la vector store filtrada
filtered_vector_store_clinical_trial.add_documents(documentos_clinical_trial)

# Escribir 5 consultas de ensayos clínicos con filtros de metadatos
consultas_clinical_trial = [
    {
        "query" : "treatment studies for breast cancer in women",
        "metadata_filter" : {
                                "conditions": "Breast Cancer",
                                "sex": "FEMALE"
                            }
    },
    {
        "query" : "Usage of AI for cancer detection",
        "metadata_filter" : {
                                
                                "conditions": "Cancer",
                                "study_type": "OBSERVATIONAL"
                            }
    },
    {
        "query" : "early phase trials for solid tumors",
        "metadata_filter" : {
                                
                                "phase": "PHASE1",
                                "study_type": "INTERVENTIONAL"
                            }
    },
    {
        "query" : "observational studies in adult patients",
        "metadata_filter" : {
                                
                                "study_type" : "OBSERVATIONAL",
                                "criteria_split_status" : "both_detected"
                            }
    },
    {
        "query" : "screening or mammography studies with healthy volunteers",
        "metadata_filter" : {
                                
                                "healthy_volunteers": "True"
                            }
    }
]


# Ejecutamos las búsquedas y mostramos los resultados
for i in range(len(consultas_clinical_trial)):
    query = consultas_clinical_trial[i]["query"]
    metadata_filter = consultas_clinical_trial[i]["metadata_filter"]
    print(f"\n--- Consulta {i+1}: '{query}' ---")
    
    # Buscamos los 5 documentos más relevantes
    resultados = filtered_vector_store_clinical_trial.search(
                                                                query, 
                                                                top_k=5, 
                                                                metadata_filter=metadata_filter
                                                            )
    
    for rank, result in enumerate(resultados, 1):
        print(f"\n  Resultado {rank}:")
        # Mostramos Score, Texto y Metadatos
        print(f"    * Score: {result.score:.4f}")
        print(f"    * Texto: {result.document.text}")
        print(f"    * Metadatos: {result.document.metadata}")



Se han agregado 60337 documentos.
Total de embeddings generados: 60337

--- Consulta 1: 'treatment studies for breast cancer in women' ---

  Resultado 1:
    * Score: 0.7061
    * Texto: Comparison of Four Different Treatment Regimens in Treating Women With Stage I Breast Cancer Protocol of a Randomized Trial for the Management of Small Well-Differentiated and Special Type Carcinomas of the Breast RATIONALE: Radiation therapy uses high-energy x-rays to damage tumor cells. Estrogen can stimulate the growth of breast cancer cells. Hormone therapy using tamoxifen may fight breast cancer by blocking the uptake of estrogen by the tumor cells. Combining radiation therapy and tamoxifen with surgery may kill more tumor cells. It is not yet known which treatment regimen is most effective for stage I breast cancer.
PURPOSE: Randomized phase III trial to compare the effectiveness of four different treatment regimens in treating women who have stage I breast cancer. Breast Cancer tamoxifen citrat

## Reflexiones

Esta actividad fue fundamental para comprender verdaderamente cómo funciona la búsqueda semántica "por debajo del capó". Al implementar las clases `VectorStore` y `FilteredVectorStore` desde cero, sin depender de librerías de alto nivel como FAISS o LangChain, pude experimentar y entender el flujo interno: desde la representación de texto en vectores densos (*embeddings*), hasta el cálculo matemático de la similitud del coseno para clasificar y devolver los documentos más relevantes.

Además, el uso de buenas prácticas de **Programación Orientada a Objetos (POO)** hizo que el desarrollo fuera mucho más eficiente. Al aplicar la **herencia**, la clase `FilteredVectorStore` pudo reutilizar los métodos de inicialización (`__init__`) y de carga de datos (`add_documents`) de su clase padre (`VectorStore`). Esto evitó la duplicación de código y permitió que la clase derivada se enfocara exclusivamente en sobrescribir el método `search` para incorporar la lógica de evaluación de diccionarios para el filtrado exacto.

Finalmente, el ejercicio demostró el poder de combinar búsquedas semánticas con filtros de metadatos. Mientras que la similitud del coseno agrupa conceptos afines, los metadatos imponen reglas duras (como filtrar por enfermedad o fase de un ensayo clínico), lo que da como resultado un sistema de recuperación de información altamente preciso y aplicable a problemas del mundo real.